# 02 — Avaliação dos modelos

Reproduz o split e os modelos do notebook `AnaliseExploratoria.ipynb` (T05–T11) sem alterá-lo, e confere se as acurácias de validação batem com as registradas lá (Regressão Logística 97,7%, Random Forest 96,5%, KNN 97,7%). As próximas seções (T12 em diante) usam o conjunto de teste.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

## Dados e split (mesmos parâmetros do T08)

In [ ]:
df = pd.read_csv("../data/data.csv").drop(columns=['id', 'Unnamed: 32'])
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15/0.85, stratify=y_temp, random_state=42
)
print(len(X_train), len(X_val), len(X_test))

## Modelos (mesma configuração dos T09–T11)

In [ ]:
def make_model(classifier):
    preprocessor = ColumnTransformer(transformers=[
        ('num', Pipeline(steps=[('scaler', StandardScaler())]), list(X.columns))
    ])
    return Pipeline(steps=[('preprocessor', preprocessor), ('classifier', classifier)])

modelos = {
    'Regressão Logística': make_model(LogisticRegression(random_state=42)),
    'Random Forest': make_model(RandomForestClassifier(random_state=42)),
    'KNN': make_model(KNeighborsClassifier()),
}
for m in modelos.values():
    m.fit(X_train, y_train)

## Conferência com os resultados do T09–T11

In [ ]:
esperado = {'Regressão Logística': 0.977, 'Random Forest': 0.965, 'KNN': 0.977}
for nome, m in modelos.items():
    acc = accuracy_score(y_val, m.predict(X_val))
    assert round(acc, 3) == esperado[nome], (nome, acc)
    print(f"{nome:20s} acurácia (validação) = {acc:.3f}  OK")

## T12 — Avaliação no conjunto de teste

O teste só é usado aqui, uma única vez, com os modelos treinados apenas no treino. A classe positiva é **M (maligno)**: o recall dela mede quantos casos malignos o modelo detecta; um falso negativo (maligno classificado como benigno) é o erro clinicamente mais grave.

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

linhas = []
matrizes = {}
for nome, m in modelos.items():
    pred = m.predict(X_test)
    linhas.append({
        'Modelo': nome,
        'Accuracy': accuracy_score(y_test, pred),
        'Recall (M)': recall_score(y_test, pred, pos_label='M'),
        'Precisão (M)': precision_score(y_test, pred, pos_label='M'),
        'F1 (M)': f1_score(y_test, pred, pos_label='M'),
    })
    matrizes[nome] = confusion_matrix(y_test, pred, labels=['B', 'M'])

resultados = pd.DataFrame(linhas).set_index('Modelo').round(3)
resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (nome, cm) in zip(axes, matrizes.items()):
    ConfusionMatrixDisplay(cm, display_labels=['Benigno', 'Maligno']).plot(ax=ax, colorbar=False)
    ax.set_title(nome)
plt.tight_layout()
plt.savefig("../reports/figures/matrizes_confusao_teste.png", dpi=150)
plt.show()

for nome, cm in matrizes.items():
    tn, fp, fn, tp = cm.ravel()
    print(f"{nome:20s} falsos negativos (M como B) = {fn} | falsos positivos (B como M) = {fp}")

**Conclusão T12** (teste: 86 amostras, 32 malignas):

| Modelo | Accuracy | Recall (M) | Precisão (M) | F1 (M) |
|---|---|---|---|---|
| Regressão Logística | 0,988 | 0,969 | 1,000 | 0,984 |
| Random Forest | 0,965 | 0,906 | 1,000 | 0,951 |
| KNN | 0,953 | 0,906 | 0,967 | 0,935 |

- **Métrica principal: recall da classe maligna.** Um falso negativo deixa passar um tumor maligno, erro mais grave que um falso positivo, que apenas leva a um exame adicional. A acurácia sozinha esconde isso, pois a base tem ~63% de benignos.
- A Regressão Logística teve o melhor resultado em todas as métricas: 1 falso negativo e nenhum falso positivo. Random Forest e KNN erraram 3 casos malignos cada.
- **Cautela:** com apenas 32 casos malignos no teste, cada erro muda o recall em ~3 pontos percentuais; as diferenças entre modelos (1 a 3 erros) não são estatisticamente conclusivas. Validação cruzada daria uma estimativa mais estável.
- O modelo apoia a triagem; a decisão diagnóstica é sempre do médico.
